# Edge Detector from Scratch

**What you'll learn:** what convolution actually computes, how kernels work, and why CNNs start with edge detection.

**Estimated time:** 30–45 min &nbsp;|&nbsp; **Runtime:** CPU is fine

---

### The big idea before any code

A digital image is just a 2D grid of numbers (pixel intensities, 0–255). An **edge** is a place where those numbers change sharply — a boundary between light and dark.

**Convolution** is the operation that detects this: slide a small matrix (a **kernel**) over the image, computing a weighted sum at each position. The kernel's shape determines what it detects.

? **Before running anything:** If you had a 3x3 kernel where the left column is all −1 and the right column is all +1 (middle is 0), what kind of edges would it detect — horizontal or vertical? Why?


## 1  Load and inspect an image

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data as skdata
from scipy.signal import convolve2d

img = skdata.camera().astype(float)
print("Image shape:", img.shape)
print("Pixel value range:", img.min(), "to", img.max())
print("Sample 5x5 patch from the top-left corner:")
print(img[:5, :5].astype(int))

The image is a 2D array. Each number is a pixel's brightness (0 = black, 255 = white).

? **What does the 5x5 patch tell you about that corner of the image?** Look at the numbers — are they bright or dark?

Now display it:


In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(img, cmap='gray')
plt.axis('off')
plt.title('Original image (cameraman)')
plt.show()

## 2  Design a kernel yourself

Before seeing the Sobel filter, let's build intuition.

**Your task:** Fill in the `blur_kernel` — a 3x3 kernel where every entry is 1/9. This is a **box blur**: each output pixel becomes the average of its 9 neighbors. Why would averaging produce a blur?

Then fill in `edge_kernel` — try to make something that detects vertical edges by using −1 on the left column and +1 on the right column.


In [ ]:
# Box blur: average of the 3x3 neighborhood
# YOUR CODE HERE — fill a 3x3 numpy array where every entry is 1/9
blur_kernel = np.ones((3, 3)) / 9.0

# Simple vertical edge detector: -1 left column, 0 middle, +1 right column
# YOUR CODE HERE — construct this kernel
edge_kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
], dtype=float)

print("Blur kernel:")
print(blur_kernel)
print("\nEdge kernel:")
print(edge_kernel)

In [ ]:
# Apply both kernels and see the difference
blurred = convolve2d(img, blur_kernel, mode='same', boundary='symm')
edges_simple = convolve2d(img, edge_kernel, mode='same', boundary='symm')

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(blurred, cmap='gray'); axes[1].set_title('After blur kernel'); axes[1].axis('off')
axes[2].imshow(edges_simple, cmap='gray'); axes[2].set_title('After your edge kernel'); axes[2].axis('off')
plt.tight_layout(); plt.show()

? **What kind of edges did your kernel pick up?** (Vertical stripes? Horizontal? Diagonal?)

Now meet the **Sobel filter** — a smarter edge kernel that weights the center row more heavily, making it less sensitive to noise.


## 3  The Sobel filters

In [ ]:
# The Sobel kernels — study these for a moment before running
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=float)

sobel_y = sobel_x.T  # .T is the transpose: rows become columns

print("sobel_x (detects vertical edges):")
print(sobel_x)
print("\nsobel_y (detects horizontal edges):")
print(sobel_y)

? **Compare `sobel_x` to your `edge_kernel`:** What's different? Why do you think the center row is weighted x2?

Now apply both Sobel filters to the image:


In [ ]:
# YOUR CODE HERE — apply sobel_x and sobel_y using convolve2d
# (same mode and boundary='symm' as before)
gx = convolve2d(img, sobel_x, mode='same', boundary='symm')
gy = convolve2d(img, sobel_y, mode='same', boundary='symm')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(gx, cmap='gray'); axes[0].set_title('Gx — vertical edges'); axes[0].axis('off')
axes[1].imshow(gy, cmap='gray'); axes[1].set_title('Gy — horizontal edges'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## 4  Combine into a full edge map

Each Sobel filter gives us edges in one direction. To find **all** edges regardless of direction, we combine them using the **gradient magnitude**:

```
magnitude = sqrt(Gx² + Gy²)
```

This is just the Pythagorean theorem — Gx and Gy are the two components of the gradient vector, and its length (magnitude) is the edge strength.

? **Before running:** In areas with no edges, what values do Gx and Gy have? What will magnitude be there?


In [ ]:
# YOUR CODE HERE — compute the gradient magnitude
magnitude = np.sqrt(gx**2 + gy**2)

plt.figure(figsize=(7, 7))
plt.imshow(magnitude, cmap='gray')
plt.axis('off')
plt.title('Edge map (gradient magnitude)')
plt.show()

print("Max magnitude:", round(magnitude.max(), 1))
print("Mean magnitude:", round(magnitude.mean(), 1))

## 5  Implement convolution from scratch

You've been using `scipy.signal.convolve2d`. Now implement it yourself using nested loops to make sure you understand what's actually happening.

The algorithm:
- For every pixel (i, j) in the output
- Extract the 3x3 patch from `image` centered at (i, j)
- Compute the element-wise product of the patch and the kernel
- Sum those 9 products — that's the output at (i, j)

Work on a small 10x10 crop so it runs fast.


In [ ]:
def my_convolve2d(image, kernel):
    kh, kw = kernel.shape
    pad_h, pad_w = kh // 2, kw // 2
    # Pad the image so edge pixels get a full neighborhood
    padded = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='reflect')
    out = np.zeros_like(image, dtype=float)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            # YOUR CODE HERE — extract the patch and compute the dot product
            patch = padded[i:i + kh, j:j + kw]
            out[i, j] = np.sum(patch * kernel)

    return out

# Test on a small crop (full image would be slow in pure Python)
crop = img[:40, :40]
result_ours = my_convolve2d(crop, sobel_x)
result_scipy = convolve2d(crop, sobel_x, mode='same', boundary='reflect')

# They should match closely (small floating-point differences are OK)
diff = np.abs(result_ours - result_scipy).max()
print(f"Max difference vs scipy: {diff:.6f}")
print("OK Matches!" if diff < 0.01 else "X Something's off — check your patch extraction")

## 6  Apply to your own image

Upload an image from your computer (or use a Colab-provided URL) and run the full pipeline on it.


In [ ]:
# Option A: upload your own image
# from google.colab import files
# uploaded = files.upload()
# from PIL import Image
# import io
# your_img = np.array(Image.open(io.BytesIO(list(uploaded.values())[0])).convert('L'), dtype=float)

# Option B: use another skimage built-in image (uncomment one)
# your_img = skdata.astronaut()[:,:,0].astype(float)  # grayscale astronaut
your_img = skdata.coins().astype(float)

gx2 = convolve2d(your_img, sobel_x, mode='same', boundary='symm')
gy2 = convolve2d(your_img, sobel_y, mode='same', boundary='symm')
mag2 = np.sqrt(gx2**2 + gy2**2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(your_img, cmap='gray'); axes[0].set_title('Your image'); axes[0].axis('off')
axes[1].imshow(mag2, cmap='gray'); axes[1].set_title('Edge map'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## Challenge: threshold the edge map

A raw magnitude image is noisy. Convert it to a clean **binary** edge map: pixels above a threshold become white (1), below become black (0).

```python
# Try different threshold values and see what changes
threshold = 50   # experiment with 30, 50, 100, 150
binary_edges = (magnitude > threshold).astype(float)
```

? **What happens as you increase the threshold?** What are you trading off?

**The real payoff:** Every CNN's first layers are learning kernels just like these — but automatically, from data, instead of handcrafted ones. That's the connection between this notebook and the neural-network lessons.
